# Knee OA Grading — ResNet-50 Pipeline
**Phase 2 only:** Layer4 + custom head trainable from start (15.5M parameters)

In [ ]:
# ── Section 1: Imports ──────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
import time

print('All imports OK ✓')

In [ ]:
# ── Section 2: Paths and Settings ───────────────────────────────
DATA_DIR  = Path('data')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR   = DATA_DIR / 'val'
TEST_DIR  = DATA_DIR / 'test'

BATCH_SIZE  = 32
NUM_CLASSES = 5
IMG_SIZE    = 224

print('Train dir exists:', TRAIN_DIR.exists())
print('Val dir exists  :', VAL_DIR.exists())
print('Test dir exists :', TEST_DIR.exists())

In [ ]:
# ── Section 3: Class Weights ─────────────────────────────────────
# From EDA: your actual image counts per grade
TRAIN_COUNTS = {0: 2286, 1: 1046, 2: 1516, 3: 757, 4: 173}
total        = sum(TRAIN_COUNTS.values())

CLASS_WEIGHTS = torch.FloatTensor([
    total / (NUM_CLASSES * TRAIN_COUNTS[i])
    for i in range(NUM_CLASSES)
])

print('Class weights:')
for i, w in enumerate(CLASS_WEIGHTS):
    print(f'  Grade {i} → {w:.4f}')

In [ ]:
# ── Section 4: Transforms ────────────────────────────────────────
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

VAL_TEST_TRANSFORMS = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('Transforms defined ✓')

In [ ]:
# ── Section 5: DataLoaders ───────────────────────────────────────
train_dataset = ImageFolder(str(TRAIN_DIR), transform=TRAIN_TRANSFORMS)
val_dataset   = ImageFolder(str(VAL_DIR),   transform=VAL_TEST_TRANSFORMS)
test_dataset  = ImageFolder(str(TEST_DIR),  transform=VAL_TEST_TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)} images, {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} images,  {len(val_loader)} batches')
print(f'Test : {len(test_dataset)} images, {len(test_loader)} batches')

In [ ]:
# ── Section 6: Build Model ───────────────────────────────────────
# Load pretrained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Step 1: Freeze ALL layers first
for param in model.parameters():
    param.requires_grad = False

# Step 2: Replace final classification head
num_features = model.fc.in_features  # 2048
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(256, NUM_CLASSES)
)

# Step 3: Unfreeze layer4 AND custom head
# This is Phase 2 directly — no Phase 1 needed
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f'Total parameters     : {total_params:,}')
print(f'Frozen parameters    : {frozen_params:,}  (layers 1-3, frozen)')
print(f'Trainable parameters : {trainable_params:,}  (layer4 + head)')
print()
print('Architecture:')
print('  Layers 1-3 → FROZEN (pretrained ImageNet features)')
print('  Layer 4    → TRAINABLE (specializes for knee X-rays)')
print('  Head       → TRAINABLE (Linear→ReLU→Dropout→Linear(5))')

In [ ]:
# ── Section 7: Loss, Optimizer, Scheduler ───────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001   # small LR — layer4 has pretrained weights
)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f'Device    : {device}')
print(f'Loss      : CrossEntropyLoss with class weights')
print(f'Optimizer : Adam lr=0.0001')
print(f'Scheduler : ReduceLROnPlateau (patience=3, factor=0.5)')

In [ ]:
# ── Section 8: Training Functions ───────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = outputs.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f'  Batch {batch_idx+1}/{len(loader)} '
                  f'| Loss: {running_loss/(batch_idx+1):.4f} '
                  f'| Acc: {100.*correct/total:.1f}%')

    return running_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs      = model(images)
            loss         = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

print('Training functions defined ✓')

In [ ]:
# ── Section 9: Training Loop ─────────────────────────────────────
EPOCHS           = 15
best_val_loss    = float('inf')
patience_counter = 0
EARLY_STOP       = 7

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss'  : [], 'val_acc'  : []
}

print(f'Training on  : {device}')
print(f'Epochs       : {EPOCHS}')
print(f'Train images : {len(train_loader.dataset)}')
print(f'Val images   : {len(val_loader.dataset)}')
print(f'Trainable    : 15,490,565 params (layer4 + head)')
print('─' * 55)

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    print(f'\nEpoch {epoch}/{EPOCHS}')

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}%')
    print(f'  Val Loss  : {val_loss:.4f} | Val Acc  : {val_acc:.1f}%')
    print(f'  Time      : {time.time()-start:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'outputs/resnet50_best.pth')
        print(f'  ✓ Best model saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{EARLY_STOP})')
        if patience_counter >= EARLY_STOP:
            print('\nEarly stopping triggered.')
            break

print(f'\nTraining complete. Best val loss: {best_val_loss:.4f}')

In [ ]:
# ── Section 10: Training Curves ──────────────────────────────────
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_loss'], color='#534AB7', linewidth=2, label='Train Loss')
axes[0].plot(epochs_ran, history['val_loss'],   color='#D85A30', linewidth=2, label='Val Loss')
axes[0].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], color='#534AB7', linewidth=2, label='Train Acc')
axes[1].plot(epochs_ran, history['val_acc'],   color='#D85A30', linewidth=2, label='Val Acc')
axes[1].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('ResNet-50 Training — Layer4 + Head Fine-tuning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/training_curves.png')

In [ ]:
# ── Section 11: Evaluation on Test Set ──────────────────────────
model.load_state_dict(torch.load('outputs/resnet50_best.pth'))
model.eval()

all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print('── Classification Report ──')
print(classification_report(
    all_labels, all_preds,
    target_names=['Grade 0','Grade 1','Grade 2','Grade 3','Grade 4']
))

auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
print(f'AUC-ROC (macro ovr): {auc:.4f}')

In [ ]:
# ── Section 12: Confusion Matrix ────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    cm,
    display_labels=['G0 Normal','G1 Doubtful','G2 Mild','G3 Moderate','G4 Severe']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/confusion_matrix.png')

In [ ]:
# ── Section 13: GradCAM ──────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model.load_state_dict(torch.load('outputs/resnet50_best.pth'))
model.eval()

target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

def generate_gradcam(image_tensor, pred_label):
    targets       = [ClassifierOutputTarget(pred_label)]
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0), targets=targets)[0]
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = image_tensor.permute(1, 2, 0).numpy()
    img  = np.clip(std * img + mean, 0, 1).astype(np.float32)
    overlay = show_cam_on_image(img, grayscale_cam, use_rgb=True)
    return img, grayscale_cam, overlay

# Find one sample per grade across test batches
grade_names = {0:'Normal', 1:'Doubtful', 2:'Mild', 3:'Moderate', 4:'Severe'}
shown = {}

for images_batch, labels_batch in test_loader:
    for i in range(len(labels_batch)):
        grade = labels_batch[i].item()
        if grade not in shown:
            shown[grade] = images_batch[i]
    if len(shown) == 5:
        break

print('Found grades:', sorted(shown.keys()))

fig, axes = plt.subplots(5, 3, figsize=(12, 22))

for row, grade in enumerate(sorted(shown.keys())):
    img_tensor = shown[grade]
    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0).to(device))
        pred   = output.max(1)[1].item()

    original, heatmap, overlay = generate_gradcam(img_tensor, pred)

    axes[row][0].imshow(original, cmap='gray')
    axes[row][0].set_title(f'Original\nTrue: Grade {grade} ({grade_names[grade]})',
                           fontsize=9, fontweight='bold')
    axes[row][0].axis('off')

    axes[row][1].imshow(heatmap, cmap='jet')
    axes[row][1].set_title(f'GradCAM Heatmap\nPred: Grade {pred} ({grade_names[pred]})',
                           fontsize=9, fontweight='bold')
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].set_title('Overlay on X-ray', fontsize=9, fontweight='bold')
    axes[row][2].axis('off')

plt.suptitle('GradCAM — What the model attends to per KL grade\nRed/Yellow = high attention regions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/gradcam_per_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/gradcam_per_grade.png')